In [ ]:
import os
import sys
import io
import contextlib

import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start):
    """Walk upward from `start` to the nearest ancestor containing both greedy/ and core/."""
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(d, 'greedy')) and os.path.isdir(os.path.join(d, 'core')):
            return d
        d = os.path.dirname(d)
    raise RuntimeError('could not find a directory containing greedy/ and core/ above ' + start)


ROOT = find_project_root(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from greedy.greedy import greedy_trajectory
from plotting.heatmap_plot import plot_trajectory_density_grid
from core.cache import cached_run

CACHE_DIR = os.path.join(ROOT, '.cache', 'three_targets')

In [ ]:
def run_trial(seed, gamma, A, src, N, K, T, sigma, stride, beta=1.0, r_stop=None, R_frac=0.20):
    """One greedy run for a single seed -> decimated, arrival-trimmed (x, y) points.

    Each cell's path is trimmed to its own first-arrival step (+ a short dwell
    buffer) so post-arrival equilibrium jitter doesn't dominate the density.

    `beta` is the kernel bandwidth (P_ij = exp(-beta * d^gamma)); beta=1 is the
    base model.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        result = greedy_trajectory(src, A, gamma, N, K, T, mode='exp', log=True,
                                    sigma=sigma, seed=seed, r_stop=r_stop, beta=beta)
    traj = result['trajectory']                                     # (K+1, N, 2)

    arm_dist = np.linalg.norm(A[:, None] - A[None, :], axis=-1)
    R = R_frac * arm_dist[arm_dist > 0].mean()                       # "arrived" radius

    dist_to_A = np.linalg.norm(traj[:, :, None, :] - A[None, None, :, :], axis=-1)
    arrived = dist_to_A.min(axis=2) < R                              # (K+1, N)

    K1 = traj.shape[0]
    first_arrival = np.where(arrived.any(axis=0), arrived.argmax(axis=0), K1 - 1)
    dwell = max(1, int(0.03 * K1))
    cutoff = np.minimum(first_arrival + dwell, K1 - 1)
    keep = np.arange(K1)[:, None] <= cutoff[None, :]

    return traj[keep][::stride].astype(np.float32)

In [ ]:
# parameters (topology_atlas/shape_grids.py recipe, unchanged from the source script)
N, DT, T0, SIGMA, GAMMA = 40, 0.02, 8.0, 0.01, 1.5
BETA = 1.6         # kernel bandwidth: P_ij = exp(-BETA * d^gamma). BETA=1 -> base model.
T = T0 * N
K = int(round(T / DT))

b, h = 0.9, 2.5
SRC = np.array([0.0, -h])
R_STOP = 0.05 if GAMMA < 1 else None


def geom3(ratio):
    """Base pair at +-b on the x-axis, apex at height ratio*b above the origin."""
    return np.array([[-b, 0.0], [b, 0.0], [0.0, ratio * b]])


RATIOS = [1.0, 1.5, 1.73, 2.5]
LABELS = ['1.0 (stalled)', '1.5 (transitional)', '1.73 (equilateral)', '2.5 (elongated)']
N_SEEDS = 30
STRIDE = 10

CMAP = 'viridis'
GAMMA_POW = 0.45   # PowerNorm exponent: boosts faint single-path trails
BINS = 220

In [ ]:
# run the trial over every e/b ratio x seed combination (120 greedy runs total -- cached after the first run)
params = dict(N=N, K=K, T=T, SIGMA=SIGMA, GAMMA=GAMMA, BETA=BETA, STRIDE=STRIDE, R_STOP=R_STOP,
              b=b, h=h, RATIOS=RATIOS, N_SEEDS=N_SEEDS, SRC=SRC.tolist())


def _compute_pts_by_ratio():
    pts = {}
    for ratio in RATIOS:
        A = geom3(ratio)
        pts[ratio] = np.concatenate(
            [run_trial(seed, GAMMA, A, SRC, N, K, T, SIGMA, STRIDE, beta=BETA, r_stop=R_STOP) for seed in range(N_SEEDS)],
            axis=0,
        )
        print(f'e/b={ratio}, beta={BETA}: {pts[ratio].shape[0]} points from {N_SEEDS} seeds')
    return pts


pts_by_ratio = cached_run(CACHE_DIR, 'pts_by_ratio', params, _compute_pts_by_ratio)

In [ ]:
lim_points = [geom3(ratio) for ratio in RATIOS] + [SRC[None, :]]

fig, axes = plot_trajectory_density_grid(
    pts_by_ratio, RATIOS, [f'e/b = {label}' for label in LABELS],
    lim_points=lim_points, bins=BINS, cmap=CMAP, gamma_pow=GAMMA_POW, pad=1.22,
    figsize_per_panel=(5.2, 6.0), rotate=True,
    suptitle=rf'k = 3 topologies across arrangement shape e/b  ($\gamma$={GAMMA:g}, $\beta$={BETA:g})',
    # caption=(f'b={b}, h={h}, $\\gamma$={GAMMA} fixed, $\\beta$={BETA:g}, N={N}, {N_SEEDS} seeds per panel. e/b=1: base '
    #          'and apex nearly collide, most cells stall before reaching any target. '
    #          'e/b=1.73 (equilateral): every arm reached, arm brightness even. e/b=2.5: '
    #          'base pair close, apex far -- arm brightness visibly unequal.'),
    savefig='density_3_k3_topologies.png',
)
plt.show()